# Paper figures: "Dynamics of Null and Electrostatic Blind Spots for Quantitative PFM"

This notebook regenerates **all Euler–Bernoulli (model) figures** in

> M. Checa, H. Swicegood, R. Millan-Solsona, R. Bulanadi, J. Zhao, J. Lasseter, S. Jesse and L. Collins,
> *ACS Nano* (2026). [arXiv:2607.27486](https://arxiv.org/abs/2607.27486)

using the two-segment solver in `eb_cr_afm`. Finite-element (COMSOL) and experimental figures are not produced here.
Run it top to bottom (it takes several minutes); 300-dpi PNG and PDF files are written to `results/manuscript_figures/`.

| Output | Figure |
|---|---|
| `fig1_concept` | Fig. 1 b–e: concept panels (slope/displacement, on/off resonance, piezo vs electrostatic) |
| `fig2_maps` | Fig. 2: frequency–position maps for four contact stiffnesses |
| `fig3_null_migration` | Fig. 3: D-NS / D-ESBS migration with load for three lateral-stiffness models |
| `fig4_eb_composite` | Euler–Bernoulli composite (spectral maps, profiles, null migration) |
| `figS6_tip_height` | Fig. S6: tip-height dependence |
| `figS8_schematic` … `figS11_dc_ramp` | Figs. S8–S11: model schematic, solver validation, finite-element reconciliation, DC-bias ramp |

Geometries and excitations follow Tables S1 and S2 of the Supporting Information.


In [ ]:
# --- shared setup -----------------------------------------------------------
import os, sys, numpy as np
sys.path.insert(0, os.path.join("..", "src"))
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar
from eb_cr_afm import (CantileverParams, ContactParams, ExcitationParams,
                       SolverOptions, solve_single_frequency, frequency_sweep)
from eb_cr_afm.solver import (compute_alpha4, compute_contact_stiffnesses,
                              compute_auxiliary, compute_tip_forces,
                              _solve_analytical, distributed_load,
                              _solve_analytical_two_segment)

OUT = os.path.join("..", "results", "manuscript_figures")
os.makedirs(OUT, exist_ok=True)

VERM, BLUE, ORAN, GREEN, PURP, GRAY = "#D55E00", "#0072B2", "#E69F00", "#009E73", "#CC79A7", "#7F7F7F"
plt.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans", "font.size": 9, "axes.linewidth": 0.8,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True,
    "ytick.right": True, "legend.frameon": False, "legend.fontsize": 8,
    "grid.alpha": 0.22, "savefig.dpi": 300, "savefig.bbox": "tight"})

def save(fig, name):
    fig.savefig(os.path.join(OUT, name + ".png"))
    fig.savefig(os.path.join(OUT, name + ".pdf"))
    print("saved", name)

O600 = SolverOptions(n_points=600, g=100.0, method="analytical")
O800 = SolverOptions(n_points=800, g=100.0, method="analytical")
O300 = SolverOptions(n_points=300, g=100.0, method="analytical")

def S(f, cant, ct, exc, opts=O600):
    return solve_single_frequency(float(f), cant, ct, exc, opts)

def interp_c(r, xs, key="z"):
    return (CubicSpline(r["x"], np.real(r[key]))(xs)
            + 1j * CubicSpline(r["x"], np.imag(r[key]))(xs))

def find_fcr(cant, ct, exc, f_lo, f_hi, obs=None):
    """coarse sweep + sub-Hz golden refinement of the first contact resonance"""
    obs = cant.contact_x - 90e-6 if obs is None else obs
    o = SolverOptions(n_points=300, g=100.0, method="analytical",
                      f_start=f_lo, f_stop=f_hi, f_step=2e3)
    sw = frequency_sweep(cant, ct, exc, o, obs_x=obs)
    f0 = sw["freqs"][int(np.argmax(sw["amplitudes_disp"]))]
    res = minimize_scalar(lambda f: -S(f, cant, ct, exc, O300)["A_disp"],
                          bounds=(f0 - 2.5e3, f0 + 2.5e3), method="bounded",
                          options={"xatol": 0.5})
    return float(res.x)

def null_disp(r, cant, win=70e-6, thresh=0.05, nres=20001):
    """near-tip displacement null: (x, depth, interior?)"""
    x = r["x"]; m = x >= cant.L - win
    xs = np.linspace(x[m][0], x[-1], nres)
    A = np.abs(interp_c(r, xs))
    i = int(np.argmin(A))
    return xs[i], A[i] / A.max(), (5 < i < nres - 6 and A[i] / A.max() < thresh)

def null_slope(r, cant, lo=0.10, hi=0.98, thresh=0.02, nres=20001):
    """interior slope null: (x, interior?)"""
    from scipy.signal import argrelmin
    x = r["x"]
    m = (x > lo * cant.L) & (x < hi * cant.L)
    xs = np.linspace(x[m][0], x[m][-1], nres)
    A = np.abs(interp_c(r, xs, "zp"))
    ref = np.max(np.abs(r["zp"]))
    idx = [i for i in argrelmin(A)[0] if A[i] < thresh * ref]
    return (xs[idx[0]], True) if idx else (np.nan, False)

def exc_pz(d1=3e-12):
    return ExcitationParams(d1=d1, d2=0, f0=0, use_f0=False, bias_coupled_f0=False,
                            delta_V=0.0, V_AC=0.0, use_distributed_load=False)

def exc_es(vac=3.0, dv=1.0):
    return ExcitationParams(d1=0, d2=0, f0=0, use_f0=False, bias_coupled_f0=False,
                            delta_V=dv, V_AC=vac, eps_r=1.0, use_distributed_load=True)

def exc_f0(f0=1e-10):
    return ExcitationParams(d1=0, d2=0, f0=f0, use_f0=True, bias_coupled_f0=False,
                            delta_V=0.0, V_AC=0.0, use_distributed_load=False)

def exc_combined(d1=3e-12, vac=3.0, dv=1.0, f0=1e-10):
    return ExcitationParams(d1=d1, d2=0, f0=f0, use_f0=True, bias_coupled_f0=False,
                            delta_V=dv, V_AC=vac, eps_r=1.0, use_distributed_load=True)

# Effective lateral contact stiffness (compliance-limited spring).
# The lateral tip-sample spring k* acts through the elastic Si tip cone, whose
# own lateral bending compliance (~3 E I_cone / H^3 for a ~12 um cone) is in
# series with the contact, so the beam never feels the full k*.  K_CONE is set
# to reproduce the COMSOL on-resonance overhang rebound (~0.17; see Fig. S10a)
# and is of the expected physical order.  k2_eff -> k* for a soft contact and
# saturates at K_CONE for a stiff one.
K_CONE = 2000.0   # N/m, tip-cone lateral bending stiffness
def k2_eff(k1):
    return 1.0 / (1.0 / k1 + 1.0 / K_CONE)

F_QS = 30e3   # quasistatic drive frequency [Hz]
print("setup OK; output dir:", os.path.abspath(OUT))


In [ ]:
# --- Table S1 geometries ------------------------------------------------------
# Figure 2: Bradler reference cantilever
G1 = dict(L=240e-6, w=40e-6, H=6e-6, k=3.0, m=5e-11, mT=5e-12, phi=0.2,
          tip_offset=15e-6)
# Figure 1 / 3 / S6 / S10 / S11: BudgetSensors Multi75E-G (SEM overhang dx=10.9 um)
G3 = dict(L=225e-6, w=28e-6, H=12e-6, k=2.8, m=5.2e-11, mT=0.0,
          phi=np.radians(11.0), tip_offset=10.9e-6)
# S9 (solver-correction demo): tall-tip geometry where the legacy fold appears
G9 = dict(L=240e-6, w=40e-6, H=15e-6, k=3.0, m=5e-11, mT=5e-12, phi=0.2,
          tip_offset=15e-6)
C1 = CantileverParams(**G1)
C3 = CantileverParams(**G3)
C9 = CantileverParams(**G9)
d_end = lambda x, cant: (cant.L - x) * 1e6     # um from cantilever end
print("contact positions (um from end):",
      d_end(C1.contact_x, C1), d_end(C3.contact_x, C3))

In [ ]:
# --- Figure 1 b-e: concept panels ---------------------------------------------
# Multi75E-G geometry (C3, same as Figs. 3/S6/S10/S11).
# Lateral contact is frictionless (k2 = 0) -- the simplest model, for the concept figure.
# With the electrostatic load on a full-width overhang, small k1 gives no
# interior D-ESBS, so k1 is chosen where one exists.
from matplotlib.colors import LinearSegmentedColormap
# Frictionless (k2 = 0) concept figure. k1 chosen for maximal D-NS/D-ESBS
# separation (a low/moderate load where the four conditions are clearly
# distinct); below k1 ~ 1250 N/m the D-ESBS runs off the free end.
K1_FIG1 = 1500.0
ct1 = ContactParams(k1=K1_FIG1, k2=0.0, c1=20e-6, c2=0.0)
fcr1 = find_fcr(C3, ct1, exc_pz(), 60e3, 900e3)
print("f_CR = %.1f kHz" % (fcr1/1e3))

def curves(f):
    rp = S(f, C3, ct1, exc_pz(), O800)
    re_ = S(f, C3, ct1, exc_es(), O800)
    rc = S(f, C3, ct1, exc_combined(), O800)
    return rp, re_, rc

r_qs = curves(F_QS)
r_on = curves(fcr1)

# amplitude-strip colormaps for piezo/electrostatic: literally the upper/lower
# halves of RdBu_r (same colormap as the ratio strip), so all three strips
# share one consistent red<->blue color language.
RDBU_R = plt.get_cmap("RdBu_r")
CMAP_PIEZO = LinearSegmentedColormap.from_list("piezo_cmap", RDBU_R(np.linspace(0.5, 1.0, 256)))
CMAP_ES = LinearSegmentedColormap.from_list("es_cmap", RDBU_R(np.linspace(0.5, 0.0, 256)))
RATIO_CLIP = 2.0   # log10 ratio color scale cap (100:1 piezo:electrostatic)
CHIP_COLOR = "#262626"   # unified label-chip background for all three strips

# larger, more legible text sizing for this figure
FS_TITLE, FS_AXLABEL, FS_TICK, FS_LEGEND, FS_NULL, FS_CHIP = 15, 12, 11, 10, 9, 9

fig = plt.figure(figsize=(9.8, 7.0))
outer_gs = fig.add_gridspec(2, 2, hspace=0.32, wspace=0.38)
panels = [
    ("off-resonance slope  ($f$ = 30 kHz)", r_qs, "zp", 0, 0, "S-ESBS"),
    ("on-resonance slope  ($f = f_{CR}$)",   r_on, "zp", 0, 1, "S-NS"),
    ("off-resonance displacement",           r_qs, "z",  1, 0, "D-ESBS"),
    ("on-resonance displacement",            r_on, "z",  1, 1, "D-NS"),
]
xs_um = np.linspace(0, C3.L * 1e6, 1500)
for title, (rp, re_, rc), key, row, col, nulname in panels:
    inner_gs = outer_gs[row, col].subgridspec(4, 1, height_ratios=[1, 0.11, 0.11, 0.11], hspace=0.12)
    ax = fig.add_subplot(inner_gs[0])
    ax_pz = fig.add_subplot(inner_gs[1], sharex=ax)
    ax_es = fig.add_subplot(inner_gs[2], sharex=ax)
    ax_rt = fig.add_subplot(inner_gs[3], sharex=ax)

    # shade the free overhang (past the interior contact) + mark the contact point,
    # so the near-tip displacement nulls (D-ESBS, D-NS) are seen sitting on the overhang
    ax.axvspan(C3.contact_x * 1e6, C3.L * 1e6, color="0.86", alpha=0.75, lw=0, zorder=0)
    ax.axvline(C3.contact_x * 1e6, color=GRAY, ls=":", lw=1.0, zorder=1)
    if row == 1:  # displacement panels: label the overhang + contact explicitly
        ax.text((C3.contact_x + C3.L) / 2 * 1e6, 0.52, "overhang", color="0.35",
                fontsize=7.5, ha="center", va="center", rotation=90, zorder=4)
        ax.annotate("contact", xy=(C3.contact_x * 1e6, 0.0),
                    xytext=(C3.contact_x * 1e6 - 22, 0.14), color=GRAY, fontsize=8,
                    ha="center", va="bottom",
                    arrowprops=dict(arrowstyle="->", color=GRAY, lw=0.9))

    norm = {}
    for r, lbl in ((rp, "piezo"), (re_, "es"), (rc, "combined")):
        A = np.abs(interp_c(r, xs_um * 1e-6, key))
        norm[lbl] = A / A.max()
    for lbl, c, lb, ls in (("piezo", "k", "piezo", "-"), ("es", VERM, "electrostatic", "-"),
                           ("combined", BLUE, "combined", "--")):
        ax.plot(xs_um, norm[lbl], ls, color=c, lw=1.5, label=lb)

    # null of the ES component (ESBS rows) / of the combined resonant (NS rows)
    if "ESBS" in nulname:
        src, srckey = re_, key
    else:
        src, srckey = rc, key
    if key == "zp":
        xn, ok = null_slope(src, C3)
    else:
        xn, dep, ok = null_disp(src, C3, thresh=0.05)
    if ok:
        ax.axvline(xn * 1e6, color=GREEN, ls="--", lw=1.2)
        # name label near the top: S-ESBS and the displacement labels sit to the
        # LEFT of the line, S-NS to the right (its interior is emptier there);
        # the numeric null position is added at the bottom, where curves -> 0
        to_left = nulname != "S-NS"
        null_ha = "right" if to_left else "left"
        null_dx = -6.0 if to_left else 6.0
        null_y = 1.0
        ax.text(xn * 1e6 + null_dx, null_y, "%s\n%.0f µm" % (nulname, xn * 1e6),
                color=GREEN, fontsize=FS_NULL, fontweight="bold", ha=null_ha, va="top",
                linespacing=1.25,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1.5))

    # log10(piezo_norm / electrostatic_norm), clipped to +-RATIO_CLIP
    ratio = np.log10(np.clip(norm["piezo"], 1e-6, None) / np.clip(norm["es"], 1e-6, None))
    ratio = np.clip(ratio, -RATIO_CLIP, RATIO_CLIP)

    strips = [(ax_pz, norm["piezo"], CMAP_PIEZO, dict(vmin=0, vmax=1), "Piezo"),
              (ax_es, norm["es"], CMAP_ES, dict(vmin=0, vmax=1), "Electrostatic"),
              (ax_rt, ratio, "RdBu_r", dict(vmin=-RATIO_CLIP, vmax=RATIO_CLIP), "Log Ratio")]
    for axx, arr, cmap, lims, label in strips:
        axx.imshow(arr[None, :], aspect="auto", cmap=cmap, extent=[0, C3.L * 1e6, 0, 1], **lims)
        axx.set_yticks([])
        axx.text(0.012, 0.5, label, transform=axx.transAxes, fontsize=FS_CHIP,
                  color="white", ha="left", va="center",
                  bbox=dict(facecolor=CHIP_COLOR, edgecolor="none",
                           boxstyle="round,pad=0.15"))
        if ok:
            axx.axvline(xn * 1e6, color=GREEN, ls="--", lw=1.0)

    ax.tick_params(labelbottom=False, labelsize=FS_TICK)
    ax_pz.tick_params(labelbottom=False)
    ax_es.tick_params(labelbottom=False)
    ax_rt.tick_params(labelbottom=True, labelsize=FS_TICK)
    ax_rt.set_xlabel(r"position ($\mu$m)", fontsize=FS_AXLABEL)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("norm. $|z'|$" if key == "zp" else "norm. $|z|$", fontsize=FS_AXLABEL)
    ax.set_title(title, loc="left", fontsize=FS_TITLE)
    ax.grid(True)
    if row == 1 and col == 0:
        ax.legend(loc="upper left", fontsize=FS_LEGEND)
fig.suptitle("Fig. 1 b-e (regenerated): Multi75E-G, $k_1$ = %g N/m, frictionless "
             "$k_2$ = 0, two-segment solver" % (K1_FIG1,),
             fontsize=11, y=0.995)
save(fig, "fig1_concept")
plt.close(fig)

In [ ]:
# --- Figure 2: frequency-position maps ----------------------------------------
# Multi75E-G geometry (C3), matching Figs. 1/3/S6/S10/S11. Driven electrostatically
# only (d1 = 0, distributed load with V_AC = 1 V, plus a tiny f0 point force).
# Blue dots mark the antiresonance trajectory; the line marks f_CR.
exc_fig2 = ExcitationParams(d1=0.0, d2=0.0, f0=1e-12, use_f0=True, bias_coupled_f0=False,
                            delta_V=1.0, V_AC=1.0, eps_r=1.0, use_distributed_load=True)
KSTARS = [100.0, 500.0, 1000.0, 10000.0]
freqs = np.linspace(10e3, 1000e3, 200)
O400 = SolverOptions(n_points=400, g=100.0, method="analytical")
maps = {}
for k1 in KSTARS:
    ct = ContactParams(k1=k1, k2=0.0, c1=20e-6, c2=0.0)
    amps_s, phs_s, amps_d, phs_d = [], [], [], []
    for f in freqs:
        r = S(f, C3, ct, exc_fig2, O400)
        amps_s.append(np.abs(r["zp"])); phs_s.append(np.angle(r["zp"], deg=True))
        amps_d.append(np.abs(r["z"]));  phs_d.append(np.angle(r["z"], deg=True))
    x_um = r["x"] * 1e6
    fcr = find_fcr(C3, ct, exc_fig2, 30e3, 950e3)
    rr = S(fcr, C3, ct, exc_fig2, O800)
    xsn, oks = null_slope(rr, C3)
    xdn, depd, okd = null_disp(rr, C3, thresh=0.05)
    maps[k1] = dict(x=x_um, As=np.array(amps_s), Ps=np.array(phs_s),
                    Ad=np.array(amps_d), Pd=np.array(phs_d), fcr=fcr,
                    xsn=xsn if oks else np.nan, xdn=xdn if okd else np.nan)
    print("k*=%g: f_CR=%.0f kHz, S-NS=%s, D-NS=%s"
          % (k1, fcr/1e3,
             "%.1f um" % (xsn*1e6) if oks else "none",
             "%.1f um" % (xdn*1e6) if okd else "none"))

from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

N_ROWS = len(KSTARS)
L_um = C3.L * 1e6
tip_um = C3.contact_x * 1e6
x_ticks = [0, 50, 100, 150, 200, int(round(L_um))]
f_ticks = [10, 1000]
cols = [("As", "log", "o", "Slope amplitude"), ("Ps", "phase", "o", "Slope phase"),
        ("Ad", "log", "D", "Displacement amplitude"), ("Pd", "phase", "D", "Displacement phase")]

fig, axes = plt.subplots(N_ROWS, 4, figsize=(22, 2.8 * N_ROWS + 1.4), sharex=True)
fig.subplots_adjust(top=0.94, bottom=0.19, left=0.07, right=0.99, hspace=0.10, wspace=0.06)

for row, k1 in enumerate(KSTARS):
    M = maps[k1]
    for col, (key, kind, marker, _) in enumerate(cols):
        ax = axes[row, col]
        Z = M[key]
        if kind == "log":
            L10 = np.log10(np.maximum(Z, 1e-300))
            vmax = np.nanpercentile(L10, 99)
            ax.pcolormesh(M["x"], freqs / 1e3, L10, cmap="coolwarm",
                         vmin=vmax - 5, vmax=vmax, shading="auto")
            # antiresonance trajectory (per-position amplitude minimum below ~1.4 f_CR)
            fm = freqs <= min(1.45 * M["fcr"], freqs[-1])
            sub = Z[fm, :]
            jmin = np.argmin(sub, axis=0)
            depth = sub[jmin, np.arange(sub.shape[1])] / sub.max(axis=0)
            good = (depth < 0.15) & (jmin > 1) & (jmin < sub.shape[0] - 2)
            ax.plot(M["x"][good], freqs[fm][jmin[good]] / 1e3, ".", color=BLUE,
                    ms=1.0, alpha=0.7)
        else:
            ax.pcolormesh(M["x"], freqs / 1e3, Z, cmap="plasma",
                         vmin=-180, vmax=180, shading="auto")
        ax.axhline(M["fcr"] / 1e3, color="w", ls=":", lw=0.7)
        ax.axvline(tip_um, color="k", lw=1.8, ls=":", alpha=0.9)
        xn = M["xsn"] if key in ("As", "Ps") else M["xdn"]
        if np.isfinite(xn):
            ax.scatter(xn * 1e6, M["fcr"] / 1e3, color="lime", s=60,
                      marker=marker, zorder=5)
        ax.set_xlim(0, L_um)
        ax.set_ylim(freqs[0] / 1e3, freqs[-1] / 1e3)
        ax.set_xticks(x_ticks)
        ax.set_yticks(f_ticks)
        ax.tick_params(length=3, labelsize=15)
        if row == N_ROWS - 1:
            ax.set_xticklabels([str(t) for t in x_ticks[:-1]] + ["%d" % L_um], fontsize=15)
            ax.set_xlabel(r"Position ($\mu$m)", fontsize=18)
        if col == 0:
            ax.set_yticklabels([str(f) for f in f_ticks], fontsize=15)
            ax.set_ylabel("Frequency (kHz)", fontsize=18)
        else:
            ax.set_yticklabels([])

    if row == 0:
        for col in (0, 2):
            axes[0, col].text(tip_um, freqs[-1] / 1e3 * 0.82, "Tip\n%d $\\mu$m" % tip_um,
                              ha="center", va="top", fontsize=15, color="k",
                              bbox=dict(facecolor="white", edgecolor="0.5", alpha=0.88,
                                       pad=2, boxstyle="round,pad=0.3"))
    axes[row, 0].text(0.02, 0.97, "$k_1$ = %g N/m" % k1, transform=axes[row, 0].transAxes,
                      va="top", ha="left", fontsize=15, color="white", fontweight="bold")

for col, (_, _, _, title) in enumerate(cols):
    axes[0, col].set_title(title, fontsize=17, fontweight="bold", pad=5)

cb_defs = [
    (ScalarMappable(norm=Normalize(-5, 0), cmap="coolwarm"),
     [-5, -2.5, 0], ["-5", "-2.5", "0"], r"$\log_{10}$ slope amp. (rel. to peak)"),
    (ScalarMappable(norm=Normalize(-180, 180), cmap="plasma"),
     [-180, 0, 180], ["-180°", "0°", "180°"], "Slope phase (°)"),
    (ScalarMappable(norm=Normalize(-5, 0), cmap="coolwarm"),
     [-5, -2.5, 0], ["-5", "-2.5", "0"], r"$\log_{10}$ disp. amp. (rel. to peak)"),
    (ScalarMappable(norm=Normalize(-180, 180), cmap="plasma"),
     [-180, 0, 180], ["-180°", "0°", "180°"], "Displacement phase (°)"),
]
cb_h, cb_y0 = 0.030, 0.075
for col, (sm, ticks, ticklabels, label) in enumerate(cb_defs):
    sm.set_array([])
    pos = axes[-1, col].get_position()
    cax = fig.add_axes([pos.x0, cb_y0, pos.width, cb_h])
    cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cb.set_ticks(ticks)
    cb.set_ticklabels(ticklabels, fontsize=15)
    cb.set_label(label, fontsize=17, labelpad=3)

fig.suptitle(r"Fig. 2 (regenerated): Multi75E-G spatial-spectral maps (electrostatic drive)  |  "
             r"$\bullet$ S-NS   $\diamondsuit$ D-NS  |  dotted white = $f_{CR}$, "
             r"dotted black = tip, blue = antiresonance",
             fontsize=17, y=0.998)
save(fig, "fig2_maps")
plt.close(fig)

In [ ]:
# --- Figure 3: D-NS vs D-ESBS migration (Multi75E-G geometry) ------------------
# Three lateral-contact models: frictionless (k2=0), the physically-correct
# compliance-limited spring (k2 = series(k*, k_cone) = k2_eff), and the rigid-tip
# limit (k2 = k*).  Only the unphysical rigid limit sustains a D-NS/D-ESBS
# separation at high stiffness; the physical model converges, so the persistent
# separation seen in COMSOL (Fig. 4) is a genuinely 3-D electrostatic effect
# that a 1-D beam cannot reproduce (see Sec. S8, Fig. S10).
K3 = np.logspace(np.log10(300.0), 6.0, 14)

def sweep_nulls(k2_of_k1):
    dns, esbs = [], []
    flo, fhi = 60e3, 700e3
    for k1 in K3:
        ct = ContactParams(k1=k1, k2=k2_of_k1(k1), c1=0.0, c2=0.0)
        fcr = find_fcr(C3, ct, exc_pz(), flo, fhi)
        flo, fhi = max(40e3, fcr - 40e3), min(900e3, fcr + 50e3)
        xn, dep, ok = null_disp(S(fcr, C3, ct, exc_pz(), O800), C3)
        dns.append(d_end(xn, C3) if ok else np.nan)
        xe, depe, oke = null_disp(S(F_QS, C3, ct, exc_es(1.0, 1.0), O800), C3)
        esbs.append(d_end(xe, C3) if oke else np.nan)
    return np.array(dns), np.array(esbs)

dns0, esbs0 = sweep_nulls(lambda k1: 0.0)         # frictionless
dnsc, esbsc = sweep_nulls(lambda k1: k2_eff(k1))  # compliance-limited (physical)
dnsk, esbsk = sweep_nulls(lambda k1: k1)          # rigid-tip limit k2 = k*
print("k2=0    D-NS:", np.round(dns0, 2)); print("k2=0    D-ESBS:", np.round(esbs0, 2))
print("k2_eff  D-NS:", np.round(dnsc, 2)); print("k2_eff  D-ESBS:", np.round(esbsc, 2))
print("k2=k*   D-NS:", np.round(dnsk, 2)); print("k2=k*   D-ESBS:", np.round(esbsk, 2))

fig = plt.figure(figsize=(10.2, 3.1))
gs = GridSpec(1, 6, width_ratios=[1, 1, 1, 1, 1.6, 1.6], wspace=0.45)
# (a) profiles at four stiffnesses, compliance-limited lateral spring
for i, k1 in enumerate([750.0, 3e3, 1e4, 1e5]):
    ax = fig.add_subplot(gs[0, i])
    ct = ContactParams(k1=k1, k2=k2_eff(k1), c1=0.0, c2=0.0)
    fcr = find_fcr(C3, ct, exc_pz(), 60e3, 700e3)
    xs = np.linspace(0, C3.L, 1200)
    ron = S(fcr, C3, ct, exc_pz(), O800)
    rqs = S(F_QS, C3, ct, exc_es(1.0, 1.0), O800)
    Aon = np.abs(interp_c(ron, xs)); Aqs = np.abs(interp_c(rqs, xs))
    ax.plot(xs * 1e6, Aon / Aon.max(), color=VERM, lw=1.4, label="on-res")
    ax.plot(xs * 1e6, Aqs / Aqs.max(), color=BLUE, lw=1.4, label="off-res (30 kHz ES)")
    xn, dep, ok = null_disp(ron, C3)
    if ok: ax.plot(xn * 1e6, 0, "v", color=VERM, ms=6, clip_on=False)
    xe, depe, oke = null_disp(rqs, C3)
    if oke: ax.plot(xe * 1e6, 0, "^", color=BLUE, ms=6, clip_on=False)
    ax.set_title("$k^*$ = %g N/m" % k1, fontsize=8)
    ax.set_xlabel(r"position ($\mu$m)", fontsize=8)
    ax.set_ylim(0, 1.05); ax.grid(True)
    if i == 0:
        ax.set_ylabel("norm. $|z|$"); ax.legend(fontsize=6, loc="upper left")
    ax.text(0.03, 0.02, "(a)" if i == 0 else "", transform=ax.transAxes)
# (b) migration for the physical (compliance-limited) model
ax = fig.add_subplot(gs[0, 4])
ax.semilogx(K3, dnsc, "-o", color=VERM, ms=4, lw=1.7, label="D-NS")
ax.semilogx(K3, esbsc, "-s", color=BLUE, ms=4, lw=1.7, label="D-ESBS")
ax.axhline(d_end(C3.contact_x, C3), color=GREEN, ls=":", lw=1.0)
ax.axhline(d_end(C3.contact_x - C3.H * np.tan(C3.phi), C3), color=GREEN, ls="--", lw=1.0)
ax.text(4e5, d_end(C3.contact_x, C3) + 0.3, "contact", color=GREEN, fontsize=7)
ax.set_xlabel("$k^*$ (N/m)"); ax.set_ylabel(r"null position ($\mu$m from end)")
ax.set_ylim(0, 20); ax.legend(fontsize=7.5, loc="lower right"); ax.grid(True, which="both")
ax.set_title(r"(b)  $k_2$ = series($k^*$, %g N/m)" % K_CONE, loc="left", fontsize=9)
# (c) separation vs k* for the three lateral models
ax = fig.add_subplot(gs[0, 5])
for dns, esbs, lab, c, ls in ((dns0, esbs0, "$k_2$ = 0", GRAY, "-"),
                              (dnsc, esbsc, "$k_2$ = series", GREEN, "-"),
                              (dnsk, esbsk, "$k_2$ = $k^*$ (rigid)", PURP, "--")):
    ax.semilogx(K3, np.abs(dns - esbs), ls, color=c, lw=1.8, marker="o", ms=3, label=lab)
ax.set_xlabel("$k^*$ (N/m)"); ax.set_ylabel(r"|D-NS $-$ D-ESBS| ($\mu$m)")
ax.set_ylim(0, 4); ax.legend(fontsize=7, loc="upper right"); ax.grid(True, which="both")
ax.set_title("(c)  only rigid $k_2$ sustains separation", loc="left", fontsize=9)
save(fig, "fig3_null_migration")
plt.close(fig)


In [ ]:
# --- Figure S11: probe-design parameter study (elastic-tip contact) ------------
# Sweep the three probe-geometry parameters -- tip height H, tilt phi, overhang dx
# -- and track the D-NS and D-ESBS positions (row 1) and their separation (row 2).
# Design goal: geometry that makes the two nulls COINCIDE at the tip (a single,
# well-defined null for quantitative PFM).  Uses the elastic-tip contact
# (k1 = k2 = k* through the cone k_cone_lat), matching Figs. 2/S3/S9.
# Base = Multi75E-G (H = 12 um, phi = 11 deg, dx = 10.9 um); k* = 10 kN/m.
from matplotlib.ticker import ScalarFormatter, NullLocator
KS11 = 1e4
def _geom(H_um, phi_deg, dx_um):
    return CantileverParams(L=225e-6, w=28e-6, H=H_um * 1e-6, k=2.8, m=5.2e-11, mT=0.0,
                            phi=np.radians(phi_deg), tip_offset=dx_um * 1e-6)
def _eval(cant):
    ct = ContactParams(k1=KS11, k2=KS11, c1=20e-6, c2=0.0, k_cone_lat=K_CONE)
    fcr = find_fcr(cant, ct, exc_pz(), 60e3, 850e3)
    win = (cant.L - cant.contact_x) + 6e-6           # near-tip window (avoid body nodes)
    xn, dn, okn = null_disp(S(fcr, cant, ct, exc_pz(), O600), cant, win=win, thresh=0.06)
    xe, de, oke = null_disp(S(F_QS, cant, ct, exc_es(1.0, 1.0), O600), cant, win=win, thresh=0.06)
    return (d_end(xn, cant) if okn else np.nan), (d_end(xe, cant) if oke else np.nan)
def _sweep(vals, mk):
    dns, esbs = [], []
    for v in vals:
        d, e = _eval(mk(v)); dns.append(d); esbs.append(e)
    return np.array(dns), np.array(esbs)

Hs = np.geomspace(3, 70, 16); dH, eH = _sweep(Hs, lambda H: _geom(H, 11, 10.9))
Ps = np.linspace(0, 22, 12);  dP, eP = _sweep(Ps, lambda p: _geom(12, p, 10.9))
Xs = np.linspace(2, 32, 16);  dX, eX = _sweep(Xs, lambda x: _geom(12, 11, x))
print("min separation (um): H=%.2f  phi=%.2f  dx=%.2f"
      % (np.nanmin(np.abs(dH - eH)), np.nanmin(np.abs(dP - eP)), np.nanmin(np.abs(dX - eX))))

def _pos(ax, x, dns, esbs, logx=False, diag=False, contact=None):
    ax.plot(x, dns, "-o", color=VERM, ms=4, lw=1.6, label="D-NS")
    ax.plot(x, esbs, "s", color=BLUE, ms=4, label="D-ESBS")
    seg = []
    for i in range(len(x)):
        if np.isfinite(esbs[i]):
            seg.append(i)
        else:
            if len(seg) > 1: ax.plot(x[seg], esbs[seg], "-", color=BLUE, lw=1.6)
            seg = []
    if len(seg) > 1: ax.plot(x[seg], esbs[seg], "-", color=BLUE, lw=1.6)
    if diag:
        ax.plot(x, x, color=GREEN, ls=":", lw=1.1)
    elif contact is not None:
        ax.axhline(contact, color=GREEN, ls=":", lw=1.1)
    if logx: ax.set_xscale("log")
    ax.grid(True, which="both")
def _sep(ax, x, dns, esbs, logx=False):
    s = np.abs(dns - esbs); seg = []
    for i in range(len(x)):
        if np.isfinite(s[i]):
            seg.append(i)
        else:
            if len(seg) > 1: ax.plot(x[seg], s[seg], "-D", color=PURP, ms=3.5, lw=1.6)
            seg = []
    if len(seg) > 1: ax.plot(x[seg], s[seg], "-D", color=PURP, ms=3.5, lw=1.6)
    ax.axhline(0, color="0.6", lw=0.8)
    if logx: ax.set_xscale("log")
    ax.grid(True, which="both"); ax.set_ylim(-0.3, 6)

fig, axes = plt.subplots(2, 3, figsize=(13, 6.6), sharex="col")
mu = r" ($\mu$m)"
_pos(axes[0, 0], Hs, dH, eH, logx=True, contact=10.9)
axes[0, 0].text(3.2, 11.4, "contact / tip", color=GREEN, fontsize=8)
axes[0, 0].set_ylabel(r"null position ($\mu$m from end)")
axes[0, 0].set_title(r"(a)  vs tip height $H$", loc="left", fontsize=10); axes[0, 0].set_ylim(0, 20)
_pos(axes[0, 1], Ps, dP, eP, contact=10.9)
axes[0, 1].set_title(r"(b)  vs tilt $\varphi$ (slope)", loc="left", fontsize=10); axes[0, 1].set_ylim(0, 20)
_pos(axes[0, 2], Xs, dX, eX, diag=True)
axes[0, 2].set_title(r"(c)  vs overhang $\Delta x$", loc="left", fontsize=10); axes[0, 2].set_ylim(0, 34)
axes[0, 2].text(3, 30, r"contact = $\Delta x$", color=GREEN, fontsize=8)
_sep(axes[1, 0], Hs, dH, eH, logx=True)
axes[1, 0].set_ylabel(r"|D-NS $-$ D-ESBS|" + mu)
axes[1, 0].set_xlabel(r"tip height $H$" + mu); axes[1, 0].set_title("(d)", loc="left", fontsize=10)
_sep(axes[1, 1], Ps, dP, eP)
axes[1, 1].set_xlabel(r"tilt $\varphi$ (deg)"); axes[1, 1].set_title("(e)", loc="left", fontsize=10)
axes[1, 1].text(0.4, 0.5, "coincide\nat tip", color=PURP, fontsize=7.5, va="bottom")
_sep(axes[1, 2], Xs, dX, eX)
axes[1, 2].set_xlabel(r"overhang $\Delta x$" + mu); axes[1, 2].set_title("(f)", loc="left", fontsize=10)
for a, base in zip(list(axes[0]) + list(axes[1]), [12, 11, 10.9, 12, 11, 10.9]):
    a.axvline(base, color=GRAY, ls="-", lw=0.8, alpha=0.45)
for a in (axes[0, 0], axes[1, 0]):
    a.set_xticks([3, 5, 10, 20, 50]); a.xaxis.set_major_formatter(ScalarFormatter())
    a.xaxis.set_minor_locator(NullLocator())
axes[1, 0].set_xlim(3, 70)
axes[0, 0].legend(loc="lower left", fontsize=8)
fig.suptitle(r"Fig. S11 (regenerated): probe-design study (elastic tip, $k^*$ = 10 kN/m) — D-NS and "
             r"D-ESBS vs tip height, tilt, and overhang (base = Multi75E-G, gray line)", fontsize=10.5, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save(fig, "figS6_tip_height")
plt.close(fig)


In [ ]:
# --- Figure S8: model schematic --------------------------------------------------
from matplotlib.patches import Polygon, Rectangle, FancyArrowPatch, Arc
fig, ax = plt.subplots(figsize=(7.6, 3.6))
XC_, L_, TH = 225.0, 240.0, 7.0
APEX = np.array([226.5, -42.0]); Ssl = 0.22
sample_y = lambda x: APEX[1] + Ssl * (x - APEX[0])
ax.add_patch(Rectangle((-16, -30), 16, 62, facecolor="0.85", edgecolor="k",
                       hatch="////", lw=0.8, zorder=1))
ax.text(-8, 36, "clamp", ha="center", fontsize=8)
ax.add_patch(Rectangle((0, 0), XC_, TH, facecolor="0.92", edgecolor="k", lw=0.9, zorder=2))
ax.add_patch(Rectangle((XC_, 0), L_ - XC_, TH, facecolor="#F5C6A5", edgecolor="k",
                       lw=0.9, zorder=2))
ax.add_patch(Polygon([(219, 0), (231, 0), tuple(APEX)], closed=True,
                     facecolor="0.55", edgecolor="k", lw=0.8, zorder=3))
xs_ = np.array([-20.0, 262.0]); ax.plot(xs_, sample_y(xs_), "-", color="k", lw=1.6, zorder=2)
xf = np.linspace(-20, 262, 60)
ax.fill_between(xf, sample_y(xf), sample_y(xf) - 9, color="0.82", zorder=1,
                hatch="\\\\\\\\", edgecolor="0.6", lw=0)
p0x = 258.0; p0y = sample_y(p0x)
ax.plot([p0x - 32, p0x + 8], [p0y, p0y], "--", color="0.5", lw=0.8, zorder=2)
ax.add_patch(Arc((p0x - 32, p0y), 44, 44, angle=0, theta1=0,
                 theta2=np.degrees(np.arctan(Ssl)), color="k", lw=0.9))
ax.text(p0x - 4, p0y + 3.2, r"$\varphi$", fontsize=10)
t_hat = np.array([1.0, Ssl]); t_hat /= np.linalg.norm(t_hat)
n_hat = np.array([-Ssl, 1.0]); n_hat /= np.linalg.norm(n_hat)
def zigzag(p0, u, length, amp, n=6):
    ts = np.linspace(0, length, 2 * n + 1)
    v = np.array([-u[1], u[0]])
    pts = np.array([p0 + u * t + v * amp * ((-1) ** i) * (0 < i < 2 * n)
                    for i, t in enumerate(ts)])
    return pts[:, 0], pts[:, 1]
zx, zy = zigzag(APEX, -n_hat, 16, 2.6); ax.plot(zx, zy, "-", color=GREEN, lw=1.3, zorder=4)
ax.text(APEX[0] + 3.0, APEX[1] - 19.5, "$k_1$, $c_1$", color=GREEN, fontsize=8.5)
zx, zy = zigzag(APEX, t_hat, 17, 2.6); ax.plot(zx, zy, "-", color=GREEN, lw=1.3, zorder=4)
ax.text(APEX[0] + 19, APEX[1] - 4.5, "$k_2$, $c_2$", color=GREEN, fontsize=8.5)
d_ref = 0.0 - sample_y(233.0)
for x in [16, 46, 76, 106, 136, 166, 196]:
    d = 0.0 - sample_y(x); ln = 19.0 * (d_ref / d) ** 2
    ax.add_patch(FancyArrowPatch((x, 0), (x, -ln), arrowstyle="-|>",
                                 mutation_scale=7, color=BLUE, lw=1.0, zorder=3))
for x in [229.5, 234.5, 239.0]:
    d = 0.0 - sample_y(x); ln = min(19.0 * (d_ref / d) ** 2, 0.62 * d)
    ax.add_patch(FancyArrowPatch((x, 0), (x, -ln), arrowstyle="-|>",
                                 mutation_scale=7, color=BLUE, lw=1.2, zorder=3))
ax.text(58, -24, r"$q(x)\ \propto\ w_{eff}(x)/d(x)^2$", color=BLUE, fontsize=9)
ax.text(58, -34, r"$d(x) = H\cos\varphi + (x_c-x)\sin\varphi$", color=BLUE, fontsize=8)
p_cone = np.array([222.0, -22.0])
ax.add_patch(FancyArrowPatch(tuple(p_cone), tuple(p_cone - n_hat * 15),
                             arrowstyle="-|>", mutation_scale=8, color=PURP, lw=1.4, zorder=5))
ax.text(98, -58, r"$F_{cone}$ (cone/local ES) + tilt moment", color=PURP,
        fontsize=8.5, ha="center")
ax.add_patch(FancyArrowPatch((152, -55), (218.0, -29.0), arrowstyle="-|>",
                             mutation_scale=6, color=PURP, lw=0.7, zorder=5,
                             connectionstyle="arc3,rad=-0.2"))
ax.plot([XC_, XC_], [-48, 22], ":", color=VERM, lw=1.0, zorder=2)
ax.text(97, 26.5, "segment 1  (clamped at $x=0$)", fontsize=9, ha="center")
ax.text(292, 30, "segment 2: free overhang\n" + r"$z''(L) = z'''(L) = 0$",
        fontsize=8.5, color=VERM, ha="center")
ax.add_patch(FancyArrowPatch((262, 26), (238.5, 8.5), arrowstyle="-|>",
                             mutation_scale=6, color=VERM, lw=0.8,
                             connectionstyle="arc3,rad=0.2"))
ax.text(297, -34, "at $x_c$:  $z$, $z'$ continuous\nmoment & shear jump =\n"
        "tip reaction ($T$, $X$, $U$; $R_M$, $R_F$)", fontsize=8.5, color=VERM, ha="center")
ax.add_patch(FancyArrowPatch((263, -28), (226.5, -8), arrowstyle="-|>",
                             mutation_scale=6, color=VERM, lw=0.7,
                             connectionstyle="arc3,rad=0.2"))
ax.add_patch(FancyArrowPatch((211, 0), (211, APEX[1]), arrowstyle="<|-|>",
                             mutation_scale=6, color="k", lw=0.8))
ax.text(206, -21, "$H$", fontsize=9, ha="right")
ax.add_patch(FancyArrowPatch((XC_, 13.5), (L_, 13.5), arrowstyle="<|-|>",
                             mutation_scale=6, color="k", lw=0.8))
ax.text(232.5, 16.5, r"$\Delta x$", fontsize=9, ha="center")
ax.plot([XC_, XC_], [7, 15.5], ":", color="k", lw=0.6)
ax.plot([L_, L_], [7, 15.5], ":", color="k", lw=0.6)
ax.text(214.5, -58, "$x_c$", fontsize=9, ha="right", color=VERM)
ax.set_xlim(-22, 302); ax.set_ylim(-84, 44); ax.axis("off")
fig.tight_layout()
save(fig, "figS8_schematic")
plt.close(fig)


In [ ]:
# --- Figure S9: previous vs corrected overhang treatment (+ FEM check) ---------
OPTS9 = O600

def legacy_solve(f, cant, ct, exc):
    """pre-correction behaviour: terminal BCs at x_c + cubic extrapolation,
    no ES load on the overhang"""
    omega = 2 * np.pi * f
    a4 = compute_alpha4(omega, cant, OPTS9)
    k1c, k2c = compute_contact_stiffnesses(omega, ct)
    T, X, U = compute_auxiliary(a4, k1c, k2c, cant, ct)
    Fv, Fl = compute_tip_forces(omega, k1c, k2c, exc)
    xg = np.linspace(0.0, cant.contact_x, OPTS9.n_points)
    qv = distributed_load(xg, cant, exc) if exc.use_distributed_load \
        else np.zeros(len(xg), complex)
    r = _solve_analytical(a4, qv, xg, T, X, U, Fv, Fl, cant, ct, f,
                          cant.contact_x - 90e-6)
    zc, zpc, zppc, zpppc = r["z"][-1], r["zp"][-1], r["zpp"][-1], r["zppp"][-1]
    dx = np.linspace(0.0, cant.L - cant.contact_x, 61)[1:]
    r = dict(r)
    r["x"] = np.concatenate([r["x"], cant.contact_x + dx])
    r["z"] = np.concatenate([r["z"], zc + zpc*dx + zppc/2*dx**2 + zpppc/6*dx**3])
    return r

def fem_sample(freq, cant, ct, exc, n1=450, n2=90):
    """independent Hermite-FEM forced response of the full beam"""
    omega = 2 * np.pi * freq
    a4 = compute_alpha4(omega, cant, OPTS9)
    k1c, k2c = compute_contact_stiffnesses(omega, ct)
    T, X, U = compute_auxiliary(a4, k1c, k2c, cant, ct)
    Fv, Fl = compute_tip_forces(omega, k1c, k2c, exc)
    EI, H = cant.EI, cant.H
    th = cant.phi + ct.psi
    Rm = (H/EI)*np.cos(th)*Fl - (H/EI)*np.sin(th)*Fv
    Rs = -(1.0/EI)*np.cos(th)*Fv - (1.0/EI)*np.sin(th)*Fl
    xc, L2 = cant.contact_x, cant.tip_offset
    rhoA = cant.m / cant.L
    xn = np.concatenate([np.linspace(0, xc, n1+1), xc + np.linspace(0, L2, n2+1)[1:]])
    ne = len(xn) - 1; nd = 2*(ne+1)
    K = np.zeros((nd, nd), complex); M = np.zeros((nd, nd), complex)
    for e in range(ne):
        le = xn[e+1]-xn[e]
        Ke = (EI/le**3)*np.array([[12,6*le,-12,6*le],[6*le,4*le**2,-6*le,2*le**2],
                                  [-12,-6*le,12,-6*le],[6*le,2*le**2,-6*le,4*le**2]])
        Me = (rhoA*le/420.)*np.array([[156,22*le,54,-13*le],[22*le,4*le**2,13*le,-3*le**2],
                                      [54,13*le,156,-22*le],[-13*le,-3*le**2,-22*le,4*le**2]])
        d = slice(2*e, 2*e+4); K[d, d] += Ke; M[d, d] += Me
    ic = int(np.argmin(np.abs(xn - xc))); dofs = [2*ic, 2*ic+1]
    K[np.ix_(dofs, dofs)] += EI*np.array([[U, X], [X, T]])
    F = np.zeros(nd, complex); F[2*ic] += -EI*Rs; F[2*ic+1] += EI*Rm
    D = K - (omega**2 - 1j*OPTS9.g*omega)*M
    keep = np.arange(2, nd)
    u = np.zeros(nd, complex)
    u[keep] = np.linalg.solve(D[np.ix_(keep, keep)], F[keep])
    w, thv = u[0::2], u[1::2]
    def sample(xq):
        out = np.empty(len(xq), complex)
        for j, xv in enumerate(xq):
            e = min(max(np.searchsorted(xn, xv) - 1, 0), ne - 1)
            le = xn[e+1]-xn[e]; t = (xv - xn[e]) / le
            out[j] = ((1-3*t**2+2*t**3)*w[e] + le*(t-2*t**2+t**3)*thv[e]
                      + (3*t**2-2*t**3)*w[e+1] + le*(-t**2+t**3)*thv[e+1])
        return out
    return sample

C9_ = C9
k1a = 7056.0
cta = ContactParams(k1=k1a, k2=0.85*k1a, c1=20e-6, c2=0.0)
fcra = find_fcr(C9_, cta, exc_pz(1e-12), 450e3, 620e3)
r_cor = S(fcra, C9_, cta, exc_pz(1e-12), O600)
r_leg = legacy_solve(fcra, C9_, cta, exc_pz(1e-12))
samp = fem_sample(fcra, C9_, cta, exc_pz(1e-12))
xs9 = np.linspace(0.925*C9_.L, C9_.L, 900)
x_fem = np.linspace(0.925*C9_.L, C9_.L, 26)
A0c = np.max(np.abs(r_cor["z"])); A0l = np.max(np.abs(r_leg["z"]))
A0f = np.max(np.abs(samp(np.linspace(0, C9_.L, 400))))

# branch sweep (fold region + full range)
K9 = np.unique(np.concatenate([np.logspace(3, np.log10(3e6), 16),
                               np.linspace(4500, 11000, 7)]))
leg_k, leg_x, cor_k, cor_x, leg_sk, leg_sx = [], [], [], [], [], []
prev_l = prev_c = None
flo, fhi = 380e3, 560e3
from scipy.signal import argrelmin
def deep_minima(r, cant, win=70e-6, thresh=0.03):
    x = r["x"]; m = x >= cant.L - win
    xs = np.linspace(x[m][0], x[-1], 30001)
    A = np.abs(interp_c(r, xs))
    return [(xs[i], A[i]/A.max()) for i in argrelmin(A)[0] if A[i]/A.max() < thresh]
for k1 in K9:
    ct = ContactParams(k1=k1, k2=0.85*k1, c1=20e-6, c2=0.0)
    fcr = find_fcr(C9_, ct, exc_pz(1e-12), flo, fhi)
    flo, fhi = max(80e3, fcr - 25e3), min(700e3, fcr + 30e3)
    ml = deep_minima(legacy_solve(fcr, C9_, ct, exc_pz(1e-12)), C9_)
    mc = deep_minima(S(fcr, C9_, ct, exc_pz(1e-12), O600), C9_)
    if ml:
        xl = (min(ml, key=lambda c: abs(c[0]-prev_l))[0] if prev_l is not None
              else min(v[0] for v in ml))
        leg_k.append(k1); leg_x.append(xl / C9_.L); prev_l = xl
        for v in ml:
            if abs(v[0] - xl) > 2e-8:
                leg_sk.append(k1); leg_sx.append(v[0] / C9_.L)
    if mc:
        xc_ = (min(mc, key=lambda c: abs(c[0]-prev_c))[0] if prev_c is not None
               else min(v[0] for v in mc))
        cor_k.append(k1); cor_x.append(xc_ / C9_.L); prev_c = xc_

fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9))
ax = axes[0]
ax.plot(xs9/C9_.L, np.abs(interp_c(r_leg, xs9))/A0l, "--", color=GRAY, lw=1.6,
        label="previous (terminal BC + cubic)")
ax.plot(xs9/C9_.L, np.abs(interp_c(r_cor, xs9))/A0c, "-", color=VERM, lw=1.8,
        label="two-segment (corrected)")
ax.plot(x_fem/C9_.L, np.abs(samp(x_fem))/A0f, "o", mfc="none", mec="k", ms=4.5,
        mew=0.9, label="independent FEM")
ax.axvline(C9_.contact_x/C9_.L, color=GREEN, ls=":", lw=1.0)
ax.set_xlabel("$x/L$"); ax.set_ylabel(r"$|z(x)|/|z|_{max}$")
ax.set_ylim(0, 0.06); ax.set_xlim(0.925, 1.0)
ax.legend(loc="upper left", fontsize=7)
ax.set_title("(a)  on-resonance tip profile, $k_1$=7.1 kN/m, $k_2$=0.85$k_1$",
             loc="left", fontsize=8.5)
ax = axes[1]
ax.semilogx(leg_k, leg_x, "--", color=GRAY, lw=1.4, label="previous (artifact fold)")
if leg_sk:
    ax.plot(leg_sk, leg_sx, "s", mfc="none", mec=GRAY, ms=4, label="previous 2nd nodes")
ax.semilogx(cor_k, cor_x, "-", color=VERM, lw=1.9, label="corrected")
ax.axhline(C9_.contact_x/C9_.L, color=GREEN, ls=":", lw=1.0)
ax.text(1.3e3, C9_.contact_x/C9_.L + 0.001, "contact point", color=GREEN, fontsize=7.5)
ax.set_xlabel("$k_1$ (N/m)"); ax.set_ylabel("D-NS position $x/L$")
ax.set_ylim(0.930, 1.0); ax.legend(loc="upper right", fontsize=7)
ax.set_title("(b)  D-NS branch, $k_2$ = 0.85$k_1$", loc="left", fontsize=8.5)
for a in axes: a.grid(True, which="both")
fig.tight_layout()
save(fig, "figS9_solver_correction")
plt.close(fig)


In [ ]:
# --- Figure S9d: interior equivalence of two-segment vs terminal-BC solution -----
# The two-segment extension leaves the lever-body response essentially unchanged vs
# the terminal-BC + cubic-extrapolation (Bradler-style) treatment, and differs only
# over the last ~5% near the tip -- so the overhang is immaterial for lever-body
# observables (Bradler's regime) but decisive for the near-tip nulls. Multi75E-G
# (C3, SEM geometry), frictionless, k* = 10 kN/m.  legacy_solve() is from the S9 cell.
from scipy.optimize import minimize_scalar as _mss
KS9D = 1e4
ct9d = ContactParams(k1=KS9D, k2=0.0, c1=0.0, c2=0.0)
OBS9D = C3.contact_x - 90e-6

def _twoseg9d(f):
    r = solve_single_frequency(float(f), C3, ct9d, exc_pz(), O800, obs_x=OBS9D)
    return r["x"], r["z"], abs(r["A_disp"])
def _cubic9d(f):
    r = legacy_solve(f, C3, ct9d, exc_pz())
    return r["x"], r["z"], abs(r["A_disp"])
def _fcr9d(solver):
    fr = np.linspace(200e3, 900e3, 400); A = np.array([solver(f)[2] for f in fr])
    f0 = fr[int(np.argmax(A))]
    return float(_mss(lambda f: -solver(float(f))[2], bounds=(f0 - 3e3, f0 + 3e3),
                      method="bounded").x)

f2_9d = _fcr9d(_twoseg9d); fc_9d = _fcr9d(_cubic9d)
x2, z2, _ = _twoseg9d(f2_9d); xc9, zc9, _ = _cubic9d(fc_9d)
xg9 = np.linspace(0, C3.L, 1400)
p2 = np.interp(xg9, x2, np.abs(z2)); p2 /= p2.max()
pc = np.interp(xg9, xc9, np.abs(zc9)); pc /= pc.max()
maxdev9d = np.max(np.abs(p2 - pc)[xg9 / C3.L < 0.90])
print("Fig S9d: f_CR two-seg=%.2f kHz, cubic=%.2f kHz (%.3f%%); "
      "max interior |d|z|| (x/L<0.9)=%.3f%%"
      % (f2_9d / 1e3, fc_9d / 1e3, 100 * abs(f2_9d - fc_9d) / f2_9d, 100 * maxdev9d))

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.4))
ax[0].plot(xg9 / C3.L, p2, "-", color=VERM, lw=1.8, label="two-segment")
ax[0].plot(xg9 / C3.L, pc, "--", color=BLUE, lw=1.5, label="terminal BC + cubic ext.")
ax[0].axvline(C3.contact_x / C3.L, color=GREEN, ls=":", lw=1.0)
ax[0].axvspan(0.90, 1.0, color="0.92", zorder=0)
ax[0].text(0.945, 0.55, "near-tip", fontsize=7, ha="center", color="0.4")
ax[0].set_xlabel("$x/L$"); ax[0].set_ylabel("norm. $|z(x)|$ (on resonance)")
ax[0].set_ylim(-0.03, 1.08); ax[0].legend(fontsize=7.5, loc="upper left"); ax[0].grid(alpha=0.3)
ax[0].set_title("(a)  interior overlap, near-tip divergence", loc="left", fontsize=9)

fr9 = np.linspace(f2_9d - 40e3, f2_9d + 40e3, 500)
A2 = np.array([_twoseg9d(f)[2] for f in fr9]); Ac = np.array([_cubic9d(f)[2] for f in fr9])
ax[1].plot(fr9 / 1e3, A2 / A2.max(), "-", color=VERM, lw=1.8, label="two-segment")
ax[1].plot(fr9 / 1e3, Ac / Ac.max(), "--", color=BLUE, lw=1.5, label="terminal BC + cubic ext.")
ax[1].set_xlabel("frequency (kHz)"); ax[1].set_ylabel("norm. $|z|$ at mid-lever (0.53 L)")
ax[1].legend(fontsize=7.5); ax[1].grid(alpha=0.3)
ax[1].set_title("(b)  mid-lever spectrum: essentially identical", loc="left", fontsize=9)
fig.suptitle("Fig. S9d: two-segment vs terminal-BC overhang treatment "
             "($k^*$ = 10 kN/m, Multi75E-G, $k_2$ = 0)", fontsize=9)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save(fig, "figS9d_interior_equivalence")
plt.close(fig)


In [ ]:
# --- Figure S10: reconciliation (shape fingerprint + separation is 3-D) --------
# (a) on-resonance overhang rebound |z(L)|/max vs lateral stiffness k2 at two
#     contact stiffnesses, against the COMSOL rebound band -> identifies the
#     physical (compliance-limited) lateral spring, k2 ~ K_CONE.
# (b) D-NS / D-ESBS separation vs k* for the physical vs rigid lateral models
#     (from the Fig. 3 sweep): the physical model converges while only the
#     unphysical rigid limit reproduces a persistent separation -> the COMSOL
#     separation is a 3-D electrostatic effect, not a lateral-stiffness effect.
def rebound_vs_k2(kstar, k2vals):
    out = []
    for k2 in k2vals:
        ct = ContactParams(k1=kstar, k2=k2, c1=0.0, c2=0.0)
        fcr = find_fcr(C3, ct, exc_pz(), 60e3, 900e3)
        r = S(fcr, C3, ct, exc_pz(), O800)
        A = np.abs(r["z"]); out.append(A[-1] / A.max())
    return np.array(out)

K2V = np.geomspace(200.0, 2e5, 16)
reb5 = rebound_vs_k2(1e5, K2V)
reb6 = rebound_vs_k2(1e6, K2V)
print("rebound(k*=1e5):", np.round(reb5, 3))
print("rebound(k*=1e6):", np.round(reb6, 3))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
ax = axes[0]
ax.axhspan(0.14, 0.20, color=BLUE, alpha=0.12, zorder=0)
ax.text(2.3e2, 0.205, "COMSOL rebound band", fontsize=7, color=BLUE)
ax.semilogx(K2V, reb5, "-o", color=VERM, ms=3.5, lw=1.6, label="$k^*$ = $10^5$ N/m")
ax.semilogx(K2V, reb6, "-s", color=ORAN, ms=3.5, lw=1.6, label="$k^*$ = $10^6$ N/m")
ax.axvline(K_CONE, color=GREEN, ls="--", lw=1.2)
ax.text(K_CONE * 1.15, 0.30, "$k_{cone}$\n= %g N/m" % K_CONE, color=GREEN, fontsize=7)
ax.set_xlabel("lateral stiffness $k_2$ (N/m)")
ax.set_ylabel("on-res overhang rebound  |z(L)|/max")
ax.set_ylim(0, 0.35); ax.legend(fontsize=7.5, loc="upper right"); ax.grid(True, which="both")
ax.set_title("(a)  mode-shape fingerprint fixes $k_2$", loc="left", fontsize=9)
ax = axes[1]
for dns, esbs, lab, c, ls in ((dnsc, esbsc, "$k_2$ = series (physical)", GREEN, "-"),
                              (dnsk, esbsk, "$k_2$ = $k^*$ (rigid tip)", PURP, "--")):
    ax.semilogx(K3, np.abs(dns - esbs), ls, color=c, lw=1.8, marker="o", ms=3, label=lab)
ax.axhspan(2.5, 3.5, color=BLUE, alpha=0.12, zorder=0)
ax.text(4e2, 3.05, "COMSOL separation", fontsize=7, color=BLUE)
ax.set_xlabel("$k^*$ (N/m)"); ax.set_ylabel(r"|D-NS $-$ D-ESBS| ($\mu$m)")
ax.set_ylim(0, 4); ax.legend(fontsize=7.5, loc="center right"); ax.grid(True, which="both")
ax.set_title("(b)  physical model converges; separation is 3-D", loc="left", fontsize=9)
fig.tight_layout()
save(fig, "figS10_reconciliation")
plt.close(fig)


In [ ]:
# --- Figure S10b (optional comparison): explicit elastic-tip contact -----------
# Instead of the scalar compliance-limited spring k2_eff, model the tip cone
# explicitly: raw isotropic contact springs (k1 = k2 = k*, the FEM condition)
# acting through an elastic cone of lateral stiffness K_CONE at the apex
# (ContactParams.k_cone_lat; see solver.compute_effective_contact).  Beyond the
# series softening captured by k2_eff, this adds the second role of cone
# elasticity: releasing the rigid-tip kinematic lock z = H tan(theta) z', so
# the near-tip nulls become drive-dependent.  Result: the D-ESBS moves to the
# TIP side of the D-NS (the FEM ordering, with no overhang-width tuning) and
# the separation passes through the COMSOL band at experimentally relevant
# stiffness; at extreme k* the axially rigid cone re-pins the beam and the 1-D
# separation closes, so the fully persistent FEM offset at k* >= 1e5 requires
# the 3-D field loading the cone (see Sec. S8).
dnsE, esbsE, rebE = [], [], []
flo, fhi = 60e3, 700e3
for k1 in K3:
    ctE = ContactParams(k1=k1, k2=k1, c1=0.0, c2=0.0, k_cone_lat=K_CONE)
    fcr = find_fcr(C3, ctE, exc_pz(), flo, fhi)
    flo, fhi = max(40e3, fcr - 40e3), min(900e3, fcr + 50e3)
    rn = S(fcr, C3, ctE, exc_pz(), O800)
    A = np.abs(rn["z"]); rebE.append(A[-1] / A.max())
    xn, dep, ok = null_disp(rn, C3)
    dnsE.append(d_end(xn, C3) if ok else np.nan)
    xe, depe, oke = null_disp(S(F_QS, C3, ctE, exc_es(1.0, 1.0), O800), C3)
    esbsE.append(d_end(xe, C3) if oke else np.nan)
dnsE, esbsE, rebE = map(np.array, (dnsE, esbsE, rebE))
print("elastic tip  D-NS:   ", np.round(dnsE, 2))
print("elastic tip  D-ESBS: ", np.round(esbsE, 2))
print("elastic tip  rebound:", np.round(rebE, 3))
print("signed separation (D-NS - D-ESBS, >0 = D-ESBS on tip side):",
      np.round(dnsE - esbsE, 2))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.0))
ax = axes[0]
ax.semilogx(K3, dnsE, "-o", color=VERM, ms=4, lw=1.7, label="D-NS (elastic tip)")
ax.semilogx(K3, esbsE, "-s", color=BLUE, ms=4, lw=1.7, label="D-ESBS (elastic tip)")
ax.semilogx(K3, dnsc, "--", color=VERM, lw=1.1, alpha=0.55, label="D-NS (scalar $k_{2,eff}$)")
ax.semilogx(K3, esbsc, "--", color=BLUE, lw=1.1, alpha=0.55, label="D-ESBS (scalar $k_{2,eff}$)")
ax.axhline(d_end(C3.contact_x, C3), color=GREEN, ls=":", lw=1.0)
ax.text(4e5, d_end(C3.contact_x, C3) + 0.4, "contact", color=GREEN, fontsize=7)
ax.set_xlabel("$k^*$ (N/m)"); ax.set_ylabel(r"null position ($\mu$m from end)")
ax.set_ylim(0, 20); ax.legend(fontsize=6.2, loc="lower right"); ax.grid(True, which="both")
ax.text(2.6e4, 13.7, "scalar $k_{2,eff}$ limit\n(pivots to contact + $H$ tan φ)",
        fontsize=6, color="0.45", ha="center", va="bottom")
ax.set_title("(a)  explicit elastic tip: $k_1$=$k_2$=$k^*$ + cone", loc="left", fontsize=9)
ax = axes[1]
ax.axhspan(2.5, 3.5, color=BLUE, alpha=0.12, zorder=0)
ax.text(1.2e4, 3.05, "COMSOL separation", fontsize=7, color=BLUE, ha="center")
ax.semilogx(K3, dnsE - esbsE, "-o", color=ORAN, ms=4, lw=1.8,
            label="elastic tip (signed; >0 = tip side)")
ax.semilogx(K3, np.abs(dnsc - esbsc), "-", color=GREEN, lw=1.4, label="scalar $k_{2,eff}$")
ax.axhline(0.0, color="0.6", lw=0.8)
ax.set_xlabel("$k^*$ (N/m)"); ax.set_ylabel(r"D-NS $-$ D-ESBS ($\mu$m)")
ax.set_ylim(-1, 9); ax.legend(fontsize=6.8, loc="upper right"); ax.grid(True, which="both")
ax.annotate("1-D separation closes\n($\\propto 1/k^*$; see caption)", xy=(4e5, 0.12),
            xytext=(7e4, 3.4), fontsize=6.5, color="0.4", ha="center",
            arrowprops=dict(arrowstyle="->", color="0.5", lw=0.8))
ax.set_title("(b)  the elastic cone puts the D-ESBS on the tip side", loc="left", fontsize=9)
fig.tight_layout()
save(fig, "figS10b_elastic_tip")
plt.close(fig)


In [ ]:
# --- Figure S11: locating the D-ESBS by DC-bias ramp (model MEFM) --------------
xsA = np.linspace(C3.L - 45e-6, C3.L - 0.2e-6, 4000)
def PE(k1):
    # full elastic-tip contact (isotropic k2=k1 through the cone k_cone),
    # matching Fig. 2 -- releases the rigid z=H tan(theta) z' lock so the
    # nulls collapse to the CONTACT point (not contact + H tan(phi)).
    ct = ContactParams(k1=k1, k2=k1, c1=20e-6, c2=0.0, k_cone_lat=K_CONE)
    P = interp_c(S(F_QS, C3, ct, exc_pz(), O800), xsA)
    E = interp_c(S(F_QS, C3, ct, exc_es(1.0, 1.0), O800), xsA)
    return ct, P, E

fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.1))
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
k1a_ = 5e3          # lowered from 3e5: at high k1 D-NS and D-ESBS converge;
                    # this compliance-limited load separates them (~3.9 um, COMSOL-like)
ct, P, E = PE(k1a_)
cmap = plt.get_cmap("coolwarm"); norm = Normalize(-6, 6)
ax = axes[0]
for v in [-6, -3, 0, 3, 6.0]:
    ax.plot(d_end(xsA, C3), np.abs(P + E * v) * 1e12, color=cmap(norm(v)), lw=1.5)
sw = np.abs(P[None, :] + E[None, :] * np.linspace(-6, 6, 25)[:, None])
x_esbs_ = xsA[int(np.argmin(sw.max(0) - sw.min(0)))]
fcr_ = find_fcr(C3, ct, exc_pz(), 60e3, 900e3)
rres = S(fcr_, C3, ct, exc_pz(), O800)
x_dns_ = xsA[int(np.argmin(np.abs(interp_c(rres, xsA))))]
ax.axvline(d_end(x_esbs_, C3), color=GREEN, lw=1.1)
ax.axvline(d_end(x_dns_, C3), color=VERM, ls="--", lw=1.1)
ax.axvline(d_end(C3.contact_x, C3), color=GRAY, ls=":", lw=1.0)
ax.text(d_end(x_esbs_, C3) + 0.4, ax.get_ylim()[1] * 0.9, "D-ESBS\n(DC pinch)",
        color=GREEN, fontsize=7.5, va="top", ha="right")
ax.text(d_end(x_dns_, C3) - 0.4, ax.get_ylim()[1] * 0.55, "D-NS", color=VERM,
        fontsize=8, ha="right")
ax.set_xlim(45, 0); ax.set_ylim(bottom=0)
ax.set_xlabel(r"distance from cantilever end ($\mu$m)")
ax.set_ylabel("|z| (pm)")
ax.set_title(r"(a)  DC ramp at $k_1 = 5{\times}10^3$ N/m (elastic-tip contact)", loc="left", fontsize=9)
sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02, fraction=0.05)
cb.set_label(r"$V_{DC}-V_{CPD}$ (V)", fontsize=8); cb.ax.tick_params(labelsize=7)

K11 = np.logspace(np.log10(3e2), 6, 12)
esbs11, dns11, dep11, has11 = [], [], [], []
for k1 in K11:
    ct, P, E = PE(k1)
    sw = np.abs(P[None, :] + E[None, :] * np.linspace(-6, 6, 25)[:, None])
    swing = sw.max(0) - sw.min(0)
    i = int(np.argmin(swing))
    esbs11.append(d_end(xsA[i], C3))
    dep11.append(swing[i] / np.median(np.abs(P)))
    has11.append(5 < i < len(xsA) - 5)
    fcr_ = find_fcr(C3, ct, exc_pz(), 60e3, 900e3)
    rres = S(fcr_, C3, ct, exc_pz(), O800)
    dns11.append(d_end(xsA[int(np.argmin(np.abs(interp_c(rres, xsA))))], C3))
esbs11, dns11, dep11 = map(np.array, (esbs11, dns11, dep11))
has11 = np.array(has11)
good = has11 & (dep11 < 0.10)
ax = axes[1]
ax.semilogx(K11[good], esbs11[good], "-o", color=BLUE, lw=1.7, ms=4,
            label="D-ESBS (usable interior null)")
ax.semilogx(K11[~good], esbs11[~good], "x", color=BLUE, ms=5, mew=1.3,
            label="no interior null (min at free end)")
ax.semilogx(K11, dns11, "-", color=VERM, lw=1.7, label="D-NS")
ax.axhline(d_end(C3.contact_x, C3), color=GRAY, ls=":", lw=1.0)
# mark where panel (a) sits on this sweep
ax.axvline(k1a_, color="0.5", ls="-", lw=0.8, alpha=0.6)
ax.text(k1a_*1.15, 15.4, "panel (a)", fontsize=6.5, color="0.4", rotation=90, va="top")
if good.any():
    ax.axvspan(K11[0], K11[good][0], color="0.92", zorder=0)
    ax.text(K11[0]*1.3, 6.5, "no positional\nES blind spot", fontsize=7, color="0.4")
ax.set_xlabel("$k_1$ (N/m)"); ax.set_ylabel(r"null position ($\mu$m from end)")
ax.set_ylim(0, 16); ax.legend(loc="lower right", fontsize=7)
ax.set_title("(b)  MEFM D-ESBS vs D-NS (elastic-tip contact)", loc="left", fontsize=9)
for a in axes: a.grid(True, which="both")
fig.tight_layout()
save(fig, "figS11_dc_ramp")
plt.close(fig)


In [ ]:
# --- Fig. 4 (EB composite): COMSOL Fig. 4-style spectra + profiles + migration -
# Combines the Fig. 2 spectral maps and the Fig. 3 profiles/migration into one
# COMSOL-Fig.4-style panel, with the PHASE spectra added (which COMSOL omits).
# Uses the elastic-tip contact (isotropic k1=k2=k* through the tip cone k_cone),
# the truest 1-D analog of COMSOL's isotropic springs.
from matplotlib.gridspec import GridSpecFromSubplotSpec
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

KS_C = [1e2, 1e3, 1e4, 1e6]           # match COMSOL Fig. 4 (b,d,e)
ZOOM_UM = 50.0                         # near-tip zoom for the displacement maps
NF_C = 260
freqs_c = np.linspace(10e3, 1000e3, NF_C)
O_MAP = SolverOptions(n_points=320, g=100.0, method="analytical")
cRED, cBLU, cGRN = "#D62728", "#1F77B4", "#2CA02C"

def _sns(rr, thr=0.15):
    # robust S-NS = deepest interior slope minimum (marks the soft-contact
    # antiresonance, e.g. k*=1e2, that a strict threshold would miss)
    xq = np.linspace(0.10 * C3.L, 0.98 * C3.L, 20001)
    A = np.abs(interp_c(rr, xq, "zp")); ref = np.max(np.abs(rr["zp"]))
    gi = int(np.argmin(A))
    return (xq[gi], True) if A[gi] / ref < thr else (np.nan, False)

K_CONE = 2000.0   # tip-cone lateral bending stiffness (elastic-tip cap)
def ct_elastic(k1):
    return ContactParams(k1=k1, k2=k1, c1=20e-6, c2=0.0, k_cone_lat=K_CONE)
exc_map = ExcitationParams(d1=0.0, d2=0.0, f0=1e-12, use_f0=True, bias_coupled_f0=False,
                           delta_V=1.0, V_AC=1.0, eps_r=1.0, use_distributed_load=True)

# ---- spectral maps ----
mapdata = {}
for k1 in KS_C:
    ct = ct_elastic(k1)
    As, Ps, Ad, Pd = [], [], [], []
    for f in freqs_c:
        r = S(f, C3, ct, exc_map, O_MAP)
        As.append(np.abs(r["zp"])); Ps.append(np.angle(r["zp"], deg=True))
        Ad.append(np.abs(r["z"]));  Pd.append(np.angle(r["z"], deg=True))
    x_um = r["x"] * 1e6
    fcr = find_fcr(C3, ct, exc_map, 60e3, 900e3)
    rr = S(fcr, C3, ct, exc_map, O800)
    xsn, oks = _sns(rr); xdn, depd, okd = null_disp(rr, C3)
    mapdata[k1] = dict(x=x_um, As=np.array(As), Ps=np.array(Ps), Ad=np.array(Ad),
                       Pd=np.array(Pd), fcr=fcr, xsn=xsn if oks else np.nan,
                       xdn=xdn if okd else np.nan)

# ---- profiles (on-res piezo, off-res 30 kHz ES) ----
profdata = {}
for k1 in KS_C:
    ct = ct_elastic(k1)
    fcr = find_fcr(C3, ct, exc_pz(), 60e3, 900e3)
    xs = np.linspace(0, C3.L, 1400)
    ron = S(fcr, C3, ct, exc_pz(), O800)
    roff = S(F_QS, C3, ct, exc_es(1.0, 1.0), O800)
    Aon = np.abs(interp_c(ron, xs)); Aoff = np.abs(interp_c(roff, xs))
    xdn, dd, okd = null_disp(ron, C3); xde, de, oke = null_disp(roff, C3)
    profdata[k1] = dict(xs=xs, on=Aon / Aon.max(), off=Aoff / Aoff.max(),
                        dns=xdn if okd else np.nan, esbs=xde if oke else np.nan)

# ---- null migration vs k* ----
K_MIG = np.logspace(2, 6, 13)
dns_m, esbs_m = [], []
flo, fhi = 60e3, 700e3
for k1 in K_MIG:
    ct = ct_elastic(k1)
    fcr = find_fcr(C3, ct, exc_pz(), flo, fhi)
    flo, fhi = max(40e3, fcr - 40e3), min(900e3, fcr + 50e3)
    xdn, dd, okd = null_disp(S(fcr, C3, ct, exc_pz(), O800), C3)
    xde, de, oke = null_disp(S(F_QS, C3, ct, exc_es(1.0, 1.0), O800), C3)
    dns_m.append(d_end(xdn, C3) if okd else np.nan)
    esbs_m.append(d_end(xde, C3) if oke else np.nan)
dns_m, esbs_m = np.array(dns_m), np.array(esbs_m)

# ================================================================ render
L_um = C3.L * 1e6
tip_um = C3.contact_x * 1e6
FS_LETTER, FS_COLTITLE, FS_AXLAB, FS_TICK = 36, 21, 22, 17
FS_KLAB, FS_CBLAB, FS_CBTICK, FS_LEG = 21, 21, 18, 20
FS_PTITLE, FS_PAXLAB, FS_PTICK = 21, 21, 17

fig = plt.figure(figsize=(24, 11.5))
outer = fig.add_gridspec(1, 2, width_ratios=[2.0, 1.30], wspace=0.13,
                         left=0.055, right=0.99, top=0.955, bottom=0.145)

# ---- A: spectral maps ----
cols = [("As", "log", "o", "Slope amp."), ("Ps", "phase", "o", "Slope phase"),
        ("Ad", "log", "D", "Displ. amp."), ("Pd", "phase", "D", "Displ. phase")]
maps_gs = GridSpecFromSubplotSpec(len(KS_C), 4, outer[0], hspace=0.16, wspace=0.07)
axmaps = np.empty((len(KS_C), 4), dtype=object)
for row, k1 in enumerate(KS_C):
    M = mapdata[k1]
    for col, (key, kind, marker, _) in enumerate(cols):
        ax = fig.add_subplot(maps_gs[row, col]); axmaps[row, col] = ax
        Z = M[key]
        if kind == "log":
            L10 = np.log10(np.maximum(Z, 1e-300)); vmax = np.nanpercentile(L10, 99)
            ax.pcolormesh(M["x"], freqs_c / 1e3, L10, cmap="coolwarm",
                          vmin=vmax - 5, vmax=vmax, shading="auto")
        else:
            ax.pcolormesh(M["x"], freqs_c / 1e3, Z, cmap="plasma",
                          vmin=-180, vmax=180, shading="auto")
        ax.axhline(M["fcr"] / 1e3, color="w", ls=":", lw=1.0)
        ax.axvline(tip_um, color="k", ls=":", lw=1.6, alpha=0.85)
        xn = M["xsn"] if key in ("As", "Ps") else M["xdn"]
        if np.isfinite(xn):
            ax.scatter(xn * 1e6, M["fcr"] / 1e3, color=cGRN, s=95, marker=marker,
                       edgecolor="w", linewidth=1.1, zorder=5)
        elif key == "Ad":
            ax.text(0.5, 0.32, "No D-NS", transform=ax.transAxes, ha="center",
                    color="lime", fontsize=FS_KLAB, fontweight="bold")
        ax.set_ylim(freqs_c[0] / 1e3, freqs_c[-1] / 1e3); ax.set_yticks([10, 500, 1000])
        if col in (2, 3):
            ax.set_xlim(L_um - ZOOM_UM, L_um); ax.set_xticks([180, 200, 220])
        else:
            ax.set_xlim(0, L_um); ax.set_xticks([0, 100, 200])
        ax.tick_params(labelsize=FS_TICK, length=3)
        if row == 0:
            ttl = cols[col][3] + (" (tip)" if col in (2, 3) else "")
            ax.set_title(ttl, fontsize=FS_COLTITLE, fontweight="bold", pad=6)
        if row != len(KS_C) - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xlabel(r"Position ($\mu$m)", fontsize=FS_AXLAB)
        if col == 0:
            ax.set_ylabel("$k^*$=$10^{%d}$ N/m\nFreq. (kHz)" % int(round(np.log10(k1))),
                          fontsize=FS_AXLAB)
        else:
            ax.set_yticklabels([])
fig.text(0.050, 0.965, "A", fontsize=FS_LETTER, fontweight="bold")

cb_defs = [("coolwarm", None, "Slope amp. (a.u.)"),
           ("plasma", Normalize(-180, 180), "Slope phase (deg)"),
           ("coolwarm", None, "Displ. amp. (a.u.)"),
           ("plasma", Normalize(-180, 180), "Displ. phase (deg)")]
for col, (cmap, norm, label) in enumerate(cb_defs):
    pos = axmaps[-1, col].get_position(fig)
    cbw = pos.width * 0.82
    cax = fig.add_axes([pos.x0 + 0.09 * pos.width, 0.058, cbw, 0.019])
    if norm is None:
        sm = ScalarMappable(norm=Normalize(0, 1), cmap=cmap); sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cb.set_ticks([0, 1]); cb.set_ticklabels(["low", "high"], fontsize=FS_CBTICK)
    else:
        sm = ScalarMappable(norm=norm, cmap=cmap); sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation="horizontal")
        cb.set_ticks([-180, 0, 180]); cb.set_ticklabels(["-180", "0", "180"], fontsize=FS_CBTICK)
    tl = cax.get_xticklabels()
    if tl:
        tl[0].set_ha("left"); tl[-1].set_ha("right")
    cb.set_label(label, fontsize=FS_CBLAB)

# ---- B: migration (top) ; C: mode-shape profiles full + tip-zoom (bottom) ----
rcref = outer[0, 1].get_position(fig)
right = GridSpecFromSubplotSpec(2, 1, outer[1], height_ratios=[1.5, 2.15], hspace=0.42)

# B: D-NS / D-ESBS migration
ax_mig = fig.add_subplot(right[0])
ax_mig.semilogx(K_MIG, dns_m, "-o", color=cRED, ms=8, lw=2.4)
ax_mig.semilogx(K_MIG, esbs_m, "-s", color=cBLU, ms=8, lw=2.4)
for k1 in KS_C:
    ax_mig.axvline(k1, color="0.6", ls="-", lw=0.9, alpha=0.45)
ax_mig.set_xlabel("$k^*$ (N/m)", fontsize=FS_PAXLAB)
ax_mig.set_ylabel(r"Distance from end ($\mu$m)", fontsize=FS_PAXLAB)
ax_mig.set_ylim(0, 20); ax_mig.invert_yaxis()
ax_mig.grid(True, which="both", alpha=0.3); ax_mig.tick_params(labelsize=FS_PTICK)
ax_mig.legend(handles=[Line2D([0], [0], color=cRED, marker="o", ms=9, lw=2.6, label="D-NS"),
                       Line2D([0], [0], color=cBLU, marker="s", ms=9, lw=2.6, label="D-ESBS")],
              loc="lower right", ncol=2, fontsize=FS_LEG, frameon=True, framealpha=0.92,
              edgecolor="0.6", columnspacing=1.2, handlelength=2.0)
_b = ax_mig.get_position(fig)
fig.text(rcref.x0 - 0.012, _b.y1 + 0.012, "B", fontsize=FS_LETTER, fontweight="bold")

# C: mode-shape profiles; row 1 full lever, row 2 near-tip zoom
cgs = GridSpecFromSubplotSpec(2, 4, right[1], hspace=0.34, wspace=0.38)
ZOOMB = (190.0, 226.0)
def _prof(ax, P, zoom):
    ax.plot(P["xs"] * 1e6, P["off"], color=cBLU, lw=2.4)
    ax.plot(P["xs"] * 1e6, P["on"],  color=cRED, lw=2.4)
    if np.isfinite(P["esbs"]): ax.axvline(P["esbs"] * 1e6, color=cBLU, ls=":", lw=2.2)
    if np.isfinite(P["dns"]):  ax.axvline(P["dns"] * 1e6,  color=cRED, ls=":", lw=2.2)
    ax.axhline(0, color="0.6", ls="--", lw=0.9); ax.tick_params(labelsize=FS_PTICK)
    if zoom:
        ax.set_xlim(*ZOOMB); ax.set_xticks([200, 220])
        mm = (P["xs"] * 1e6 >= ZOOMB[0]) & (P["xs"] * 1e6 <= ZOOMB[1])
        yt = max(P["off"][mm].max(), P["on"][mm].max()) * 1.15
        ax.set_ylim(-0.03 * yt, yt)
    else:
        ax.set_xlim(0, L_um); ax.set_ylim(-0.05, 1.14); ax.set_xticks([0, 100, 200])
axf0 = None
for j, k1 in enumerate(KS_C):
    P = profdata[k1]
    axf = fig.add_subplot(cgs[0, j]); _prof(axf, P, False)
    axf.set_title("$k^*$=$10^{%d}$" % int(round(np.log10(k1))), fontsize=FS_PTITLE)
    if j == 0:
        axf.set_ylabel("Norm. displ.\n(full)", fontsize=FS_PAXLAB - 3); axf0 = axf
    axz = fig.add_subplot(cgs[1, j]); _prof(axz, P, True)
    if j == 0:
        axz.set_ylabel("Norm. displ.\n(tip zoom)", fontsize=FS_PAXLAB - 3)
    axz.set_xlabel(r"Position ($\mu$m)", fontsize=FS_PAXLAB - 3)
fig.text(rcref.x0 - 0.012, axf0.get_position(fig).y1 + 0.012, "C",
         fontsize=FS_LETTER, fontweight="bold")
fig.legend(handles=[Line2D([0], [0], color=cBLU, lw=3.0, label="Off-res (30 kHz)"),
                    Line2D([0], [0], color=cRED, lw=3.0, label="On-res"),
                    Line2D([0], [0], color=cBLU, ls=":", lw=2.6, label="D-ESBS"),
                    Line2D([0], [0], color=cRED, ls=":", lw=2.6, label="D-NS")],
           loc="center", bbox_to_anchor=(rcref.x0 + rcref.width / 2, 0.045), ncol=2,
           fontsize=FS_LEG, frameon=True, framealpha=0.92, edgecolor="0.6",
           columnspacing=1.3, handlelength=2.1)

save(fig, "fig4_eb_composite")
plt.close(fig)


In [ ]:
# --- manifest ------------------------------------------------------------------
import glob
print("Manuscript figure outputs:")
for f in sorted(glob.glob(os.path.join(OUT, "*.png"))):
    print("  ", os.path.basename(f), "(%.0f kB)" % (os.path.getsize(f)/1024))